# MASH analysis pipeline with posterior computation

This notebook applies a fitted MASH model (from `mash_fit`) to compute posterior quantities (posterior mean, sd, lfsr, etc.) for a list of gene-SNP effect-size chunks, and derives downstream contrast and feature-score summaries. It is designed to run posterior computation in parallel over many input chunks.

## Prerequisites

Requires the MASH model from `mash_fit` (`protocol_example.mash_model.rds`).

## Description

For each input chunk (a list of matrices `bhat`/`sbhat`/`Z`), the `posterior` workflow loads the MASH model and calls `mash_compute_posterior_matrices`. Additional workflows compute posterior contrasts between conditions and feature-level scores (meta, fine-mapped, n-significant, and p-value pairs) from the contrast results.

## Input

This notebook has two workflow targets that are run in sequence.

`posterior` (computes per-region posterior mean/covariance):
- `--mash-model`: a fitted MASH model RDS (the bare `mash` object), e.g. produced by `mash_fit`.
- `--analysis-units`: a text file whose first column lists paths to posterior-input RDS chunks (one region per line).
- `--posterior-vhat-files`: matching residual-variance (vhat) matrix RDS file(s).
- Produces a manifest file (`mash_output_list_all`, tab-separated `id path` pairs) pointing to the per-region posterior RDS outputs.

`mash_posterior_contrast` (computes contrasts from the posterior output):
- `--posterior-file`: a tab-separated manifest (`id path`) where each path is a per-region posterior RDS produced by the `posterior` step above.
- `--sum-file`: a tab-separated manifest (`id path`) where each path is the corresponding raw summary-statistics RDS for that region, containing top-level `bhat`/`sbhat` matrices (the same data used to compute the posterior). The `id` values must match between `--posterior-file` and `--sum-file`.

## Output

- Per-chunk posterior RDS files (`*.posterior.rds`) containing `PosteriorMean`, `PosteriorSD`, `lfsr`, `lfdr`, `NegativeProb`, `PosteriorCov` (effects x conditions).
- A manifest list (`mash_output_list_all`) of the per-chunk posterior files.
- Contrast and feature-score summary tables from the downstream workflows.

## Steps

The `posterior` workflow is the entry point; the remaining workflows derive contrasts and feature scores from its outputs. Toy inputs use the `protocol_example.*` prefix.

**Step 1.** Compute MASH posteriors for each input chunk listed in the analysis-units file:

**Timing** (toy chr22 dataset):
- `mash_posterior`: ~30 sec


In [ ]:
sos run pipeline/mash_posterior.ipynb posterior \
    --cwd output/mash_posterior \
    --analysis-units input/finemapping/protocol_example.analysis_units.txt \
    --mash-model input/mash/protocol_example.mash_model.rds \
    --posterior-vhat-files input/twas/protocol_example.posterior_vhat.rds \
    --data-table-name strong \
    --exclude-condition 1 3

**Step 2.** Compute posterior contrasts between conditions for the sliced data:

In [ ]:
# Step 1: compute per-region posterior mean/covariance
sos run pipeline/mash_posterior.ipynb posterior \
    --cwd output/mash_posterior \
    --analysis-units input/finemapping/protocol_example.analysis_units.txt \
    --mash-model input/mash/protocol_example.mash_model.rds \
    --posterior-vhat-files input/twas/protocol_example.posterior_vhat.rds \
    --data-table-name strong --exclude-condition 1 3

# Step 2: --posterior-file and --sum-file are tab-separated "id path" manifests
# whose ids must match. --sum-file must point to the RAW summary-stats RDS for
# each region with top-level bhat/sbhat (i.e. NOT nested under --data-table-name).
# For production multi-region runs, --posterior-file is typically
# output/mash_posterior/mash_output_list_all (auto-generated above). For this
# single-region toy example we derive --sum-file from the same posterior-input
# file used above, and build both manifests explicitly:
Rscript -e 'p <- strsplit(readLines("input/finemapping/protocol_example.analysis_units.txt")[1], "\\s+")[[1]][1]; saveRDS(readRDS(p)$strong, "output/mash_posterior/protocol_example.region1_sumstats.rds")'
printf 'region1\t%s\n' "$PWD/output/mash_posterior/cache/protocol_example.posterior_input.posterior.rds" > output/mash_posterior/posterior_manifest.txt
printf 'region1\t%s\n' "$PWD/output/mash_posterior/protocol_example.region1_sumstats.rds" > output/mash_posterior/sum_manifest.txt

sos run pipeline/mash_posterior.ipynb mash_posterior_contrast \
    --cwd output/mash_posterior \
    --posterior-file output/mash_posterior/posterior_manifest.txt \
    --sum-file output/mash_posterior/sum_manifest.txt

**Step 3.** Plot the posterior contrast results:

In [ ]:
sos run pipeline/mash_posterior.ipynb posterior_cntrast_plot \
    --cwd output/mash_posterior \
    --analysis-units input/finemapping/protocol_example.analysis_units.txt

**Step 4.** Compute meta feature scores from the contrast results:

In [ ]:
sos run pipeline/mash_posterior.ipynb feature_score_meta \
    --cwd output/mash_posterior \
    --analysis-units input/finemapping/protocol_example.analysis_units.txt \
    --posterior-file input/protocol_example.posterior.rds \
    --sum-file input/protocol_example.sumstats.rds

**Step 5.** Compute feature scores from contrast results using fine-mapped eQTL/pQTL:

In [ ]:
sos run pipeline/mash_posterior.ipynb feature_score_finemap \
    --cwd output/mash_posterior \
    --analysis-units input/finemapping/protocol_example.analysis_units.txt \
    --posterior-file input/protocol_example.posterior.rds \
    --sum-file input/protocol_example.sumstats.rds

**Step 6.** Compute n-significant feature scores from contrast results:

In [ ]:
sos run pipeline/mash_posterior.ipynb feature_score_nsig \
    --cwd output/mash_posterior \
    --analysis-units input/finemapping/protocol_example.analysis_units.txt \
    --posterior-file input/protocol_example.posterior.rds \
    --sum-file input/protocol_example.sumstats.rds

**Step 7.** Compute p-value-pair feature scores from contrast results:

In [ ]:
sos run pipeline/mash_posterior.ipynb feature_pval_pair \
    --cwd output/mash_posterior \
    --analysis-units input/finemapping/protocol_example.analysis_units.txt \
    --posterior-file input/protocol_example.posterior.rds \
    --sum-file input/protocol_example.sumstats.rds

## Command interface

In [ ]:
sos run pipeline/mash_posterior.ipynb -h

## Workflow implementation

In [ ]:
[global]
parameter: cwd = path('./output')
parameter: name = 'test'
# Conditions (contexts); order matters
parameter: cells = []
# Condition groups: replicate populations of one cell type share contrast weight
parameter: group1 = []
parameter: group2 = []
parameter: group3 = []
parameter: job_size = 1
parameter: container = ''
# Number of analysis units per job
parameter: per_chunk = 1
parameter: output_prefix = ''
parameter: output_suffix = 'all'
# Exchangable effect (EE) or exchangable z-scores (EZ)
parameter: effect_model = 'EE'
# Vhat estimate identifier (e.g. simple / identity / mle)
parameter: vhat = 'simple'
parameter: data = path("fastqtl_to_mash_output/FastQTLSumStats.mash.rds")
# Significance cutoff for the contrast summary / n-sig feature score
parameter: p_cut = 0.00001
parameter: walltime = '1h'
parameter: mem = '16G'
parameter: numThreads = 1
data = data.absolute()
cwd = cwd.absolute()
if len(output_prefix) == 0:
    output_prefix = f"{data:bn}"
vhat_data = file_target(f"{cwd:a}/{output_prefix}.{effect_model}.V_{vhat}.rds")
mash_model = file_target(f"{cwd:a}/{output_prefix}.{effect_model}.V_{vhat}.mash_model.rds")

### Posterior results

## Posterior 
take all the 13K genes,
if `slice_method = True` for those with missing conditions we just drop those corresponding rows and cols in the prior model

In [ ]:
# Compute MASH posteriors per analysis unit (one posterior RDS per region)
[posterior_1]
# File listing per-region data RDS paths (col 1); each carries bhat/sbhat matrices
parameter: analysis_units = path
# Effect-size / standard-error list elements inside each region RDS
parameter: bhat_table_name = 'bhat'
parameter: shat_table_name = 'sbhat'
# Conditions (columns) to exclude; names or 1-based indices
parameter: exclude_condition = []
posterior_input = [line.split()[0] for line in open(analysis_units).readlines() if line.strip() and not line.strip().startswith('#')]
input: posterior_input, group_by = 1
output: f"{cwd}/cache/{name}.{_input:bn}.posterior.rds"
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output:bn}'
bash: expand = "${ }", stderr = f'{_output:n}.stderr', stdout = f'{_output:n}.stdout', container = container
    Rscript code/script/pecotmr_integration/mash_posterior.R \
        --data ${_input} \
        --bhat-key ${bhat_table_name} \
        --shat-key ${shat_table_name} \
        --vhat-data ${vhat_data} \
        --mash-model ${mash_model} \
        --effect-model ${effect_model} \
        --exclude-condition "${','.join([str(x) for x in exclude_condition])}" \
        --output ${_output}

In [ ]:
# Collect the per-region posterior paths into a single list
[posterior_2]
input: group_by = "all"
output: f"{cwd}/{name}.{output_suffix}.posterior_list"
bash: expand = "${ }", stderr = f'{_output:n}.stderr', stdout = f'{_output:n}.stdout', container = container
    for f in ${_input}; do echo "$f"; done > ${_output}

## Posterior contrast
- Add group in this module
    
    e.g. add MiGA-Microglia-scRNA data to this analysis to supplement the Mic cell count shortage, but the MiGA data comes from four sites with four datasets, and here I handled it in such a way that the different Mic's were always in the same state during the deviation contrast analysis. If Mic is the one be compared, each Mic sample would be set as (n_populations - 1)/n_Mic instead of n_populations

In [ ]:
# Per-region posterior contrasts (deviation + pairwise) via mashPosteriorContrast
[mash_posterior_contrast_1]
# File listing per-region data RDS paths (same units as posterior_1)
parameter: analysis_units = path
# Effect-size list element inside each region RDS (aligned to the posterior)
parameter: orig_key = 'bhat'
# Optional file of comma-separated condition groups (one per line)
parameter: grouping_recipe = ''
contrast_units = [line.split()[0] for line in open(analysis_units).readlines() if line.strip() and not line.strip().startswith('#')]
input: contrast_units, group_by = 1
output: f"{cwd}/contrast/{name}.{_input:bn}.posterior_contrast.rds"
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output:bn}'
bash: expand = "${ }", stderr = f'{_output:n}.stderr', stdout = f'{_output:n}.stdout', container = container
    Rscript code/script/pecotmr_integration/mash_posterior_contrast.R \
        --posterior ${cwd}/cache/${name}.${_input:bn}.posterior.rds \
        --orig-data ${_input} \
        --orig-key ${orig_key} \
        --cells ${','.join(cells)} \
        --group1 "${','.join(group1)}" \
        --group2 "${','.join(group2)}" \
        --group3 "${','.join(group3)}" \
        --grouping-recipe "${grouping_recipe}" \
        --output ${_output}

In [ ]:
# Summarize contrast significance across regions -> CSV
[mash_posterior_contrast_2]
input: group_by = "all"
output: f"{cwd}/{name}.posterior_sum.csv"
bash: expand = "${ }", stderr = f'{_output:n}.stderr', stdout = f'{_output:n}.stdout', container = container
    Rscript code/script/pecotmr_integration/mash_posterior_contrast_summary.R \
        --contrast ${_input} \
        --cells ${','.join(cells)} \
        --p-cutoff ${p_cut} \
        --output ${_output}

In [ ]:
# Plot the contrast significance summary as a symmetric heatmap
[mash_posterior_contrast_3, posterior_cntrast_plot]
input: group_by = "all"
output: f"{cwd}/{name}.posterior_sum.png"
bash: expand = "${ }", stderr = f'{_output:n}.stderr', stdout = f'{_output:n}.stdout', container = container
    Rscript code/script/pecotmr_integration/mash_posterior_contrast_plot.R \
        --data ${_input} \
        --output ${_output}

### feature_score_meta
Meta approach: Meta analysis with each cell deviation contrast result of each feature (a loop for each column in deviation results). And then perform meta analysis with pairwise contrasts to get a pvalue to find to understand what the specific differences are. But there is a problem that meta analysis with so many snps would cost too much time to compute, I am trying to downsample for input, Dan suggest to LD prune with a more permissive threshold.

In [ ]:
# Feature score (meta) from per-region contrast results
[feature_score_meta_1]
# File listing per-region posterior-contrast RDS paths
parameter: contrast_units = path
# Random-effects estimator (metafor via pecotmr)
parameter: meta_method = 'REML'
feature_input = [line.split()[0] for line in open(contrast_units).readlines() if line.strip() and not line.strip().startswith('#')]
input: feature_input, group_by = per_chunk
output: f"{cwd}/feature_score_meta/cache/{name}.featurescore{_index+1}.rds"
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output:bn}'
bash: expand = "${ }", stderr = f'{_output:n}.stderr', stdout = f'{_output:n}.stdout', container = container
    Rscript code/script/pecotmr_integration/mash_feature_score.R \
        --method meta \
        --contrast ${_input} \
        --meta-method ${meta_method} \
        --output ${_output}

### feature_score_finemap
feature score with fine mapped QTL (the top 1 in each CS)

Fine-mapped approach(more recommend by Dan). pick the top SNP in each CS from each cell type fine mapped results. Then pick the most significant one from contrast result. Get the z-score from that snp, which should be the score of that cell type.

In [ ]:
# Feature score (finemap) using credible sets from fine-mapping
[feature_score_finemap_1]
parameter: contrast_units = path
# Fine-mapping table RDS with cs_order / pip / variants columns
parameter: fine_mapping = path
# Conditions to score (default: all contexts present in the contrast)
parameter: conditions = ''
feature_input = [line.split()[0] for line in open(contrast_units).readlines() if line.strip() and not line.strip().startswith('#')]
input: feature_input, group_by = per_chunk
output: f"{cwd}/feature_score_finemap/cache/{name}.featurescore{_index+1}.rds"
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output:bn}'
bash: expand = "${ }", stderr = f'{_output:n}.stderr', stdout = f'{_output:n}.stdout', container = container
    Rscript code/script/pecotmr_integration/mash_feature_score.R \
        --method finemap \
        --contrast ${_input} \
        --fine-mapping ${fine_mapping} \
        --conditions "${conditions}" \
        --output ${_output}

In [ ]:
# Merge per-chunk finemap feature scores into one table
[feature_score_finemap_2]
input: group_by = "all"
output: f"{cwd}/{name}.{step_name}.feature_score_sum.csv"
bash: expand = "${ }", stderr = f'{_output:n}.stderr', stdout = f'{_output:n}.stdout', container = container
    Rscript code/script/pecotmr_integration/mash_feature_score_merge.R \
        --scores ${_input} \
        --output ${_output}

### feature_score_nsig
Calculate the number of meaningful SNPs in each feature of each cell type (actually, ratio)

In [ ]:
# Feature score (nsig) from per-region contrast results
[feature_score_nsig_1]
# File listing per-region posterior-contrast RDS paths
parameter: contrast_units = path
# Significance cutoff for the n-significant ratio
parameter: p_cutoff = 0.00001
feature_input = [line.split()[0] for line in open(contrast_units).readlines() if line.strip() and not line.strip().startswith('#')]
input: feature_input, group_by = per_chunk
output: f"{cwd}/feature_score_nsig/cache/{name}.featurescore{_index+1}.rds"
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output:bn}'
bash: expand = "${ }", stderr = f'{_output:n}.stderr', stdout = f'{_output:n}.stdout', container = container
    Rscript code/script/pecotmr_integration/mash_feature_score.R \
        --method nsig \
        --contrast ${_input} \
        --p-cutoff ${p_cutoff} \
        --output ${_output}

### feature_pval_pair
find the specific different cell type pair

perform meta analysis with pairwise contrasts to get a pvalue to find to understand what the specific differences are.

metaanalysis return a warning as "Warning message: “Ratio of largest to smallest sampling variance extremely large. May not be able to obtain stable results.”"
Which is due to the big difference between max(pairwise_standard_errors^2) and min(pairwise_standard_errors^2) So I have delete the snps with small pairwise_standard_errors as Xuewei suggested

## Anticipated Results

Produces `protocol_example.mash_posterior.rds` with posterior effect size estimates across conditions. Expect condition-specific enrichment patterns in the Mic/Ast De_Jager cell types for the toy region.

In [ ]:
# Feature score (pval_pair) from per-region contrast results
[feature_pval_pair_1]
# File listing per-region posterior-contrast RDS paths
parameter: contrast_units = path
parameter: meta_method = 'REML'
# SE floor for the pairwise meta-analysis
parameter: se_cutoff = 0.001
feature_input = [line.split()[0] for line in open(contrast_units).readlines() if line.strip() and not line.strip().startswith('#')]
input: feature_input, group_by = per_chunk
output: f"{cwd}/feature_score_pval_pair/cache/{name}.featurescore{_index+1}.rds"
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output:bn}'
bash: expand = "${ }", stderr = f'{_output:n}.stderr', stdout = f'{_output:n}.stdout', container = container
    Rscript code/script/pecotmr_integration/mash_feature_score.R \
        --method pval_pair \
        --contrast ${_input} \
        --meta-method ${meta_method} \
        --se-cutoff ${se_cutoff} \
        --output ${_output}

In [ ]:
# Merge per-chunk feature scores into one table (meta / nsig / pval_pair)
[feature_pval_pair_2, feature_score_meta_2, feature_score_nsig_2]
input: group_by = "all"
output: f"{cwd}/{name}.{step_name}.feature_score_sum.csv"
bash: expand = "${ }", stderr = f'{_output:n}.stderr', stdout = f'{_output:n}.stdout', container = container
    Rscript code/script/pecotmr_integration/mash_feature_score_merge.R \
        --scores ${_input} \
        --output ${_output}